In [1]:
import cv2
import numpy as np

# 读取两张法线图像
normal2_path = '/home/jiahao/ipsm_relighting_v3/pred_normal_omni1.png'
# normal1_path = '/home/jiahao/ipsm_relighting_v3/outputs/interiorverse/scene_0_bp_3dgs_nv12_v0/precompute_depth_normal/normal_2.png'
normal1_path="/home/jiahao/ipsm_relighting_v3/000_normal_map.png"

normal1 = cv2.imread(normal1_path, cv2.IMREAD_UNCHANGED)[..., :3].astype(np.float32)
normal2 = cv2.imread(normal2_path, cv2.IMREAD_UNCHANGED)[..., :3].astype(np.float32)

# 将像素值从[0,255]归一化到[-1,1]
def img2normal(img):
    return img / 127.5 - 1.0

n1 = img2normal(normal1)
n2 = img2normal(normal2)

# 所有xyz轴的符号组合
signs = [
    (sx, sy, sz)
    for sx in [1, -1]
    for sy in [1, -1]
    for sz in [1, -1]
]

# 计算余弦相似度均值，找出最佳符号组合
best_score = -np.inf
best_sign = None

for sx, sy, sz in signs:
    n2_flip = n2 * np.array([sx, sy, sz])
    # 归一化
    n2_flip_norm = n2_flip / (np.linalg.norm(n2_flip, axis=2, keepdims=True) + 1e-8)
    n1_norm = n1 / (np.linalg.norm(n1, axis=2, keepdims=True) + 1e-8)
    # 计算点积（余弦相似度）
    cos_sim = np.sum(n1_norm * n2_flip_norm, axis=2)
    mean_sim = np.mean(cos_sim)
    print(f"signs {sx, sy, sz}: mean cosine similarity = {mean_sim:.4f}")
    if mean_sim > best_score:
        best_score = mean_sim
        best_sign = (sx, sy, sz)

print(f"\n最佳符号组合: {best_sign}, 平均余弦相似度: {best_score:.4f}")
# 应用最佳符号组合到n2
n2_best = n2 * np.array(best_sign)

# 反归一化到[0,255]，并转换为uint8
def normal2img(normal):
    img = ((normal + 1.0) * 127.5).clip(0, 255)
    return img.astype(np.uint8)

n2_best_img = normal2img(n2_best)

# 保存到本地
save_path = '/home/jiahao/ipsm_relighting_v3/pred_normal_omni1_best_sign.png'
cv2.imwrite(save_path, n2_best_img)
print(f"已保存最佳符号组合后的法线图像到: {save_path}")

signs (1, 1, 1): mean cosine similarity = -0.9672
signs (1, 1, -1): mean cosine similarity = -0.2254
signs (1, -1, 1): mean cosine similarity = -0.2804
signs (1, -1, -1): mean cosine similarity = 0.4614
signs (-1, 1, 1): mean cosine similarity = -0.4614
signs (-1, 1, -1): mean cosine similarity = 0.2804
signs (-1, -1, 1): mean cosine similarity = 0.2254
signs (-1, -1, -1): mean cosine similarity = 0.9672

最佳符号组合: (-1, -1, -1), 平均余弦相似度: 0.9672
已保存最佳符号组合后的法线图像到: /home/jiahao/ipsm_relighting_v3/pred_normal_omni1_best_sign.png
